# 冻结价格基线的预测偏差分解

复用三个时间折的已保存模型、输入和预测，0拟合、0回测。逐日精确重构预测，分解均值偏差及输入分布变化。相关特征贡献不视为相互独立的因果作用，不按本表删特征或改交易规则。

In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.analysis.linear_forecast_attribution import decompose_forecast
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/baseline_bias_diagnosis_20261014';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=plan['provider_uri'],region='us',kernels=1)
names=['inputs/grouped','analysis/predictions',*['models/'+fold+'_price_calendar' for fold in plan['folds']]]
objects,audit=_source(root,plan['source'],names)
results={};summaries=[];features=[];pairs=[]
for fold in plan['folds']:
    result=decompose_forecast(objects['models/'+fold+'_price_calendar'],objects['inputs/grouped'],objects['analysis/predictions'][(fold,'price_calendar')])
    results[fold]=result;summaries.append(result['summary'].assign(fold=fold));features.append(result['features'].assign(fold=fold))
    pairs.append({'fold':fold,'feature_a':'dist_high_252','feature_b':'range_pos_252','training_correlation':result['training_correlation'].loc['dist_high_252','range_pos_252'],'scored_correlation':result['scored_correlation'].loc['dist_high_252','range_pos_252']})
summary_table=pd.concat(summaries,ignore_index=True);feature_table=pd.concat(features,ignore_index=True);pair_table=pd.DataFrame(pairs)
top=feature_table.assign(abs_contribution=feature_table.mean_forecast_contribution.abs()).sort_values('abs_contribution',ascending=False).groupby(['fold','target'],sort=False).head(12)
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='frozen_price_baseline_bias_decomposition',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='inputs',source_audit=audit,plan=plan)
    rec.save_objects(artifact_path='analysis',decompositions=results,summary=summary_table,features=feature_table,correlation_pairs=pair_table)
    rec.save_objects(artifact_path='sources',analysis_code=(root/'src/analysis/linear_forecast_attribution.py').read_text())
    rec.log_metrics(registered_artifact_count=7,prediction_reconciliation_max_error=float(summary_table.reconstruction_max_abs_error.max()))
summary={'experiment':plan['experiment'],'decision':'diagnostic_complete_no_new_alpha','new_model_fits':0,'backtests':0,'goal_remains_active':True,'objective_achieved':False,'promotion_eligible':False,'forecast_bias':summary_table.to_dict('records'),'correlation_pairs':pairs,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':plan['provider_uri'],'as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/decompositions','analysis/features','sources/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':plan['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}))
summary_table.to_csv(card/'bias_summary.csv',index=False);top.to_csv(card/'top_contributions.csv',index=False);pair_table.to_csv(card/'correlation_summary.csv',index=False)
summary_table,pair_table


(                 target  scored_rows  training_rows  prediction_mean  \
 0      cash_relative_h5          246            553        -0.012581   
 1  leverage_relative_h5          246            553         0.017304   
 2      cash_relative_h5          245            805        -0.051061   
 3  leverage_relative_h5          245            805         0.095428   
 4      cash_relative_h5          940           1056        -0.000107   
 5  leverage_relative_h5          940           1056         0.002839   
 
    actual_mean      bias  training_target_mean  intercept  \
 0    -0.006967 -0.005614             -0.007350  -0.007350   
 1     0.007653  0.009650              0.004719   0.004719   
 2     0.008919 -0.059981             -0.007461  -0.007461   
 3    -0.030835  0.126263              0.006128   0.006128   
 4    -0.008518  0.008411             -0.003524  -0.003524   
 5     0.005917 -0.003078             -0.002699  -0.002699   
 
    sum_feature_mean_contributions  intercept_minus

# 研究用价格基线偏差诊断

复用了guidance_information中的三个冻结价格/事件日历基线，不重新拟合、不改变预测、不运行组合回测。全部预测的线性分解重构最大绝对误差为2.78e-16，训练样本数与拟合时一致。校验均值采用与StandardScaler一致的Float64累加；此前Float32汇总差异没有通过放宽容差绕过。

2022年的五日SOXL相对SOXX对数收益预测均值为0.09543，实际为-0.03084，偏差0.12626。其中训练目标先验相对当年实际均值的差为0.03696，特征均值变化产生的预测偏移为0.08930。这些是预测量的代数分解，不是组合净收益。

最大的两项平均贡献来自dist_high_252（+0.15240）与range_pos_252（-0.13050）。二者训练相关系数0.97355，2022年评价相关系数0.89707，不能把两项当成独立因果作用。其他主要项包括high_chop_pressure、soxx_vol20、cross_breadth_pos_60及60日收益；完整贡献留在Recorder，根目录只展示各窗口前12项。

按源特征定义，dist_high_252是SOXL收盘相对过去252日高点的距离。2022年训练范围最低-0.80382，评价最低-0.90506；37.55%的评价日落在训练范围之外，线性模型仍继续外推。range_pos_252和high_chop_pressure的评价值未越过训练最小/最大范围。因此不能把全部偏差归结为某一个越界字段。所有输出仍位于历史训练标签的极值范围内，仅裁剪预测到训练标签范围不会触发修正。

2021年两头平均偏差较小；2023以后的复用整体窗口偏差也小于2022。不同窗口市场状态不同，不能用这个对比证明模型年龄就是原因，也不能把归因表当成删除特征的收益证据。

本诊断仅定位研究用线性头的失配，不是生产模拟盘表现的完整归因，也未证明输入裁剪、更新频率或更复杂模型能产生超额。下一步可冻结一个只改变按时重训机制的对照，检验是否改善校准；若信息门通过，再进行统一IBKR Tiered十万美元、独立滑点和风险约束下的完整账户检验。不得自动改动生产策略或放宽稳定超额门槛。目标未达成，保持active。
